# 63. The Zone Picking & Pick-and-Pass Balancing Problem
## Tier 3: Artificial Bee Colony Optimization

### Problem Context

The Artificial Bee Colony (ABC) algorithm models the intelligent foraging behavior of honeybee swarms to solve the zone balancing problem. In this nature-inspired approach, each bee represents a potential solution (zone assignment configuration), and the colony collectively explores the solution space through three types of bees: employed bees (current solutions), onlooker bees (probability-based exploration), and scout bees (diversification through random search).

### Key Assumptions
- Each bee represents a complete zone assignment solution
- Solution quality is measured by composite fitness function
- Bee communication follows probabilistic selection based on fitness
- Scout bees maintain diversity by abandoning exhausted food sources
- Population balance between exploitation and exploration is crucial

### Approach (Step-by-Step)

1. **Population Initialization**: Generate random feasible solutions as initial food sources

2. **Employed Bee Phase**: Each bee modifies its solution and selects greedily

3. **Onlooker Bee Phase**: Bees select solutions probabilistically based on fitness

4. **Scout Bee Phase**: Replace abandoned solutions with new random ones

5. **Convergence Tracking**: Monitor population fitness and diversity metrics

### What to Look for in the Results
- Convergence behavior over iterations
- Population diversity and exploration-exploitation balance
- Final solution quality vs baseline methods
- Parameter sensitivity analysis

### Concrete Example

We'll implement the 6-zone, 25-order system from the source:
- 6 zones with varying capacities and processing characteristics
- 25 orders with complex zone requirement patterns
- 30 bees in the colony with balanced employed/onlooker/scout ratios
- 100 iterations for convergence analysis

In [ ]:
# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
import random
import time
import copy
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Zone:
    """Represents a picking zone in the warehouse"""
    zone_id: int
    capacity: float  # items per hour
    pick_time: float  # minutes per item
    workers: int
    
@dataclass
class Order:
    """Represents a customer order"""
    order_id: int
    items_by_zone: Dict[int, int]  # zone_id -> item_count
    priority: int  # 1=high, 2=medium, 3=low
    
@dataclass
class BeeSolution:
    """Represents a bee's solution (zone assignment configuration)"""
    assignments: Dict[int, Dict[int, int]]  # order_id -> zone_id -> time_slot
    fitness: float
    trial_count: int = 0  # Number of trials without improvement

# Define the concrete example from the source (6-zone, 25-order system)
zones = [
    Zone(1, 180, 1.8, 3),  # Zone 1: 180 items/hour, 1.8 min/item, 3 workers
    Zone(2, 150, 2.2, 2),  # Zone 2: 150 items/hour, 2.2 min/item, 2 workers
    Zone(3, 200, 1.5, 3),  # Zone 3: 200 items/hour, 1.5 min/item, 3 workers
    Zone(4, 140, 2.8, 2),  # Zone 4: 140 items/hour, 2.8 min/item, 2 workers
    Zone(5, 170, 2.0, 2),  # Zone 5: 170 items/hour, 2.0 min/item, 2 workers
    Zone(6, 160, 1.6, 3)   # Zone 6: 160 items/hour, 1.6 min/item, 3 workers
]

# Generate 25 orders with varying requirements
np.random.seed(42)  # For reproducibility
orders = []
for i in range(1, 26):
    items_by_zone = {}
    # Each order requires 2-4 zones randomly
    num_zones = np.random.randint(2, 5)
    selected_zones = np.random.choice(range(1, 7), num_zones, replace=False)
    
    for zone_id in selected_zones:
        items_by_zone[zone_id] = np.random.randint(1, 10)
    
    priority = np.random.choice([1, 2, 3], p=[0.3, 0.5, 0.2])  # Weighted priorities
    orders.append(Order(i, items_by_zone, priority))

handoff_time = 2  # minutes between zones
time_horizon = 30  # planning horizon in time slots

print(f"Zone Configuration:")
for zone in zones:
    print(f"  Zone {zone.zone_id}: {zone.capacity} items/hour, {zone.pick_time} min/item, {zone.workers} workers")

print(f"\nOrder Summary (first 10 shown):")
for order in orders[:10]:
    priority_str = {1: "High", 2: "Medium", 3: "Low"}[order.priority]
    print(f"  Order {order.order_id} ({priority_str}): {order.items_by_zone}")

print(f"\nTotal Orders: {len(orders)}")

In [ ]:
class ArtificialBeeColonyOptimizer:
    """Artificial Bee Colony algorithm for zone picking optimization"""
    
    def __init__(self, zones: List[Zone], orders: List[Order], 
                 colony_size: int = 30, max_iterations: int = 100, 
                 limit: int = 10, handoff_time: float = 2, time_horizon: int = 30):
        self.zones = zones
        self.orders = orders
        self.colony_size = colony_size
        self.max_iterations = max_iterations
        self.limit = limit  # Max trials before abandonment
        self.handoff_time = handoff_time
        self.time_horizon = time_horizon
        
        # Precompute zone capacities per time slot
        self.zone_capacity_per_slot = {
            zone.zone_id: (zone.capacity / 60) * zone.workers 
            for zone in zones
        }
        
        # Initialize population
        self.food_sources = []  # List of BeeSolution objects
        self.best_solution = None
        self.best_fitness = -float('inf')
        
        # Tracking variables
        self.fitness_history = []
        self.diversity_history = []
        self.convergence_data = []
        
    def optimize(self) -> Dict:
        """Main optimization method"""
        start_time = time.time()
        
        # Initialize population
        self._initialize_population()
        
        # Main optimization loop
        for iteration in range(self.max_iterations):
            # Employed bee phase
            self._employed_bee_phase()
            
            # Onlooker bee phase
            self._onlooker_bee_phase()
            
            # Scout bee phase
            self._scout_bee_phase()
            
            # Update best solution
            self._update_best_solution()
            
            # Track convergence
            self._track_convergence(iteration)
            
            # Progress reporting
            if iteration % 20 == 0:
                avg_fitness = np.mean([fs.fitness for fs in self.food_sources])
                print(f"Iteration {iteration}: Best Fitness = {self.best_fitness:.4f}, Average = {avg_fitness:.4f}")
        
        optimization_time = time.time() - start_time
        
        return {
            'best_solution': self.best_solution,
            'best_fitness': self.best_fitness,
            'fitness_history': self.fitness_history,
            'diversity_history': self.diversity_history,
            'convergence_data': self.convergence_data,
            'optimization_time': optimization_time,
            'final_population': self.food_sources
        }
    
    def _initialize_population(self):
        """Generate initial random solutions"""
        self.food_sources = []
        
        for i in range(self.colony_size):
            solution = self._generate_random_solution()
            fitness = self._calculate_fitness(solution)
            
            bee_solution = BeeSolution(
                assignments=solution,
                fitness=fitness,
                trial_count=0
            )
            
            self.food_sources.append(bee_solution)
            
            # Update best solution
            if fitness > self.best_fitness:
                self.best_fitness = fitness
                self.best_solution = copy.deepcopy(solution)
    
    def _generate_random_solution(self) -> Dict[int, Dict[int, int]]:
        """Generate a random feasible solution"""
        solution = {}
        
        for order in self.orders:
            solution[order.order_id] = {}
            
            # Assign time slots for each required zone
            current_time = np.random.randint(0, max(5, self.time_horizon - 10))
            
            for zone_id in sorted(order.items_by_zone.keys()):
                # Ensure time slot is within horizon
                if current_time >= self.time_horizon:
                    current_time = np.random.randint(0, 5)
                
                solution[order.order_id][zone_id] = current_time
                current_time += np.random.randint(1, 4)  # Add some spacing
        
        return solution
    
    def _calculate_fitness(self, solution: Dict[int, Dict[int, int]]) -> float:
        """Calculate composite fitness for a solution"""
        # Calculate objectives
        total_completion_time = self._calculate_total_completion_time(solution)
        workload_balance = self._calculate_workload_balance(solution)
        throughput = len(self.orders) / max(total_completion_time, 1)
        
        # Composite fitness: maximize throughput and balance, minimize time
        # Normalize and combine objectives
        time_fitness = 1 / (1 + total_completion_time / 100)  # Lower time = higher fitness
        balance_fitness = 1 / (1 + workload_balance)  # Lower imbalance = higher fitness
        throughput_fitness = min(throughput / 0.5, 1)  # Normalize throughput
        
        # Weighted combination
        fitness = 0.4 * time_fitness + 0.4 * balance_fitness + 0.2 * throughput_fitness
        
        return fitness
    
    def _calculate_total_completion_time(self, solution: Dict[int, Dict[int, int]]) -> float:
        """Calculate total completion time for all orders"""
        total_time = 0
        
        for order in self.orders:
            if order.order_id in solution:
                assignments = solution[order.order_id]
                if assignments:
                    # Find the latest time slot (completion time)
                    completion_time = max(assignments.values())
                    total_time += completion_time
        
        return total_time
    
    def _calculate_workload_balance(self, solution: Dict[int, Dict[int, int]]) -> float:
        """Calculate workload balance variance across zones"""
        zone_loads = defaultdict(float)
        
        # Calculate total processing time for each zone
        for order in self.orders:
            if order.order_id in solution:
                for zone_id, time_slot in solution[order.order_id].items():
                    if zone_id in order.items_by_zone:
                        zone = next(z for z in self.zones if z.zone_id == zone_id)
                        processing_time = order.items_by_zone[zone_id] * zone.pick_time
                        zone_loads[zone_id] += processing_time
        
        # Calculate variance
        if zone_loads:
            loads = list(zone_loads.values())
            max_load = max(loads)
            min_load = min(loads)
            if max_load > 0:
                return (max_load - min_load) / max_load
        
        return 0
    
    def _employed_bee_phase(self):
        """Each employed bee modifies its solution and selects greedily"""
        for i in range(len(self.food_sources)):
            # Generate neighbor solution
            neighbor_solution = self._generate_neighbor_solution(self.food_sources[i].assignments)
            neighbor_fitness = self._calculate_fitness(neighbor_solution)
            
            # Greedy selection
            if neighbor_fitness > self.food_sources[i].fitness:
                self.food_sources[i].assignments = neighbor_solution
                self.food_sources[i].fitness = neighbor_fitness
                self.food_sources[i].trial_count = 0
            else:
                self.food_sources[i].trial_count += 1
    
    def _generate_neighbor_solution(self, current_solution: Dict[int, Dict[int, int]]) -> Dict[int, Dict[int, int]]:
        """Generate a neighbor solution by modifying one order's assignment"""
        neighbor = copy.deepcopy(current_solution)
        
        # Select random order to modify
        if neighbor:
            order_id = random.choice(list(neighbor.keys()))
            
            # Modify one zone assignment for this order
            if neighbor[order_id]:
                zone_id = random.choice(list(neighbor[order_id].keys()))
                # Modify time slot
                current_time = neighbor[order_id][zone_id]
                new_time = max(0, min(self.time_horizon - 1, 
                                    current_time + random.randint(-3, 3)))
                neighbor[order_id][zone_id] = new_time
        
        return neighbor
    
    def _onlooker_bee_phase(self):
        """Onlooker bees select solutions probabilistically and exploit"""
        # Calculate selection probabilities based on fitness
        fitness_values = [fs.fitness for fs in self.food_sources]
        total_fitness = sum(fitness_values)
        
        if total_fitness > 0:
            probabilities = [f / total_fitness for f in fitness_values]
        else:
            probabilities = [1 / len(self.food_sources)] * len(self.food_sources)
        
        # Each onlooker bee selects a solution
        for _ in range(len(self.food_sources)):
            # Roulette wheel selection
            selected_index = self._roulette_wheel_selection(probabilities)
            
            # Generate neighbor and evaluate
            neighbor_solution = self._generate_neighbor_solution(self.food_sources[selected_index].assignments)
            neighbor_fitness = self._calculate_fitness(neighbor_solution)
            
            # Greedy selection
            if neighbor_fitness > self.food_sources[selected_index].fitness:
                self.food_sources[selected_index].assignments = neighbor_solution
                self.food_sources[selected_index].fitness = neighbor_fitness
                self.food_sources[selected_index].trial_count = 0
            else:
                self.food_sources[selected_index].trial_count += 1
    
    def _roulette_wheel_selection(self, probabilities: List[float]) -> int:
        """Probabilistic selection based on fitness proportions"""
        r = random.random()
        cumulative = 0
        
        for i, prob in enumerate(probabilities):
            cumulative += prob
            if r <= cumulative:
                return i
        
        return len(probabilities) - 1
    
    def _scout_bee_phase(self):
        """Scout bees replace abandoned solutions with new random ones"""
        for i in range(len(self.food_sources)):
            if self.food_sources[i].trial_count > self.limit:
                # Replace with new random solution
                new_solution = self._generate_random_solution()
                new_fitness = self._calculate_fitness(new_solution)
                
                self.food_sources[i] = BeeSolution(
                    assignments=new_solution,
                    fitness=new_fitness,
                    trial_count=0
                )
    
    def _update_best_solution(self):
        """Update the best solution found so far"""
        for bee_solution in self.food_sources:
            if bee_solution.fitness > self.best_fitness:
                self.best_fitness = bee_solution.fitness
                self.best_solution = copy.deepcopy(bee_solution.assignments)
    
    def _track_convergence(self, iteration: int):
        """Track convergence metrics"""
        fitness_values = [fs.fitness for fs in self.food_sources]
        
        self.fitness_history.append({
            'iteration': iteration,
            'best': max(fitness_values),
            'average': np.mean(fitness_values),
            'worst': min(fitness_values),
            'std': np.std(fitness_values)
        })
        
        # Calculate population diversity
        diversity = self._calculate_population_diversity()
        self.diversity_history.append({
            'iteration': iteration,
            'diversity': diversity
        })
        
        self.convergence_data.append({
            'iteration': iteration,
            'best_fitness': self.best_fitness,
            'avg_fitness': np.mean(fitness_values),
            'diversity': diversity
        })
    
    def _calculate_population_diversity(self) -> float:
        """Calculate population diversity using Hamming distance"""
        if len(self.food_sources) < 2:
            return 0
        
        total_distance = 0
        comparisons = 0
        
        for i in range(len(self.food_sources)):
            for j in range(i + 1, len(self.food_sources)):
                distance = self._hamming_distance(self.food_sources[i].assignments, 
                                                self.food_sources[j].assignments)
                total_distance += distance
                comparisons += 1
        
        return total_distance / comparisons if comparisons > 0 else 0
    
    def _hamming_distance(self, sol1: Dict, sol2: Dict) -> float:
        """Calculate Hamming distance between two solutions"""
        if not sol1 or not sol2:
            return 0
        
        distance = 0
        total_assignments = 0
        
        for order_id in sol1:
            if order_id in sol2:
                for zone_id in sol1[order_id]:
                    if zone_id in sol2[order_id]:
                        if sol1[order_id][zone_id] != sol2[order_id][zone_id]:
                            distance += 1
                        total_assignments += 1
        
        return distance / max(total_assignments, 1)

In [ ]:
# Create and run the ABC optimizer
abc_optimizer = ArtificialBeeColonyOptimizer(
    zones=zones,
    orders=orders,
    colony_size=30,
    max_iterations=100,
    limit=10,
    handoff_time=handoff_time,
    time_horizon=time_horizon
)

print("Starting Artificial Bee Colony Optimization...")
result = abc_optimizer.optimize()

print(f"\nOptimization completed in {result['optimization_time']:.2f} seconds")
print(f"Best fitness achieved: {result['best_fitness']:.4f}")

# Analyze the best solution
if result['best_solution']:
    best_solution = result['best_solution']
    
    # Calculate final metrics
    total_completion_time = abc_optimizer._calculate_total_completion_time(best_solution)
    workload_balance = abc_optimizer._calculate_workload_balance(best_solution)
    throughput = len(orders) / max(total_completion_time, 1)
    
    print(f"\nFinal Solution Metrics:")
    print(f"  Total completion time: {total_completion_time:.1f} minutes")
    print(f"  Workload balance variance: {workload_balance:.3f}")
    print(f"  Throughput: {throughput:.3f} orders/minute")
    
    # Calculate zone utilizations
    zone_utilizations = {}
    for zone in zones:
        total_processing = 0
        for order in orders:
            if order.order_id in best_solution and zone.zone_id in best_solution[order.order_id]:
                if zone.zone_id in order.items_by_zone:
                    total_processing += order.items_by_zone[zone.zone_id] * zone.pick_time
        
        capacity_total = abc_optimizer.zone_capacity_per_slot[zone.zone_id] * time_horizon
        zone_utilizations[zone.zone_id] = total_processing / capacity_total if capacity_total > 0 else 0
    
    print(f"\nZone Utilizations:")
    for zone_id, utilization in zone_utilizations.items():
        print(f"  Zone {zone_id}: {utilization:.1%}")
    
    # Calculate balance score
    utilizations = list(zone_utilizations.values())
    if utilizations:
        avg_util = np.mean(utilizations)
        max_util = max(utilizations)
        min_util = min(utilizations)
        print(f"\nBalance Analysis:")
        print(f"  Average utilization: {avg_util:.1%}")
        print(f"  Utilization range: {min_util:.1%} - {max_util:.1%}")
        print(f"  Balance score: {1 - (max_util - min_util):.3f} (higher is better)")

In [ ]:
# Convergence Analysis Visualization
def visualize_abc_convergence(result):
    """Create comprehensive convergence analysis visualizations"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Artificial Bee Colony Convergence Analysis', fontsize=16, fontweight='bold')
    
    # Extract data
    iterations = [h['iteration'] for h in result['fitness_history']]
    best_fitness = [h['best'] for h in result['fitness_history']]
    avg_fitness = [h['average'] for h in result['fitness_history']]
    worst_fitness = [h['worst'] for h in result['fitness_history']]
    fitness_std = [h['std'] for h in result['fitness_history']]
    diversity = [d['diversity'] for d in result['diversity_history']]
    
    # 1. Fitness Evolution
    axes[0, 0].plot(iterations, best_fitness, 'g-', linewidth=2, label='Best')
    axes[0, 0].plot(iterations, avg_fitness, 'b-', linewidth=2, label='Average')
    axes[0, 0].plot(iterations, worst_fitness, 'r-', linewidth=2, label='Worst')
    axes[0, 0].fill_between(iterations, avg_fitness, worst_fitness, alpha=0.3, color='lightblue')
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel('Fitness')
    axes[0, 0].set_title('Population Fitness Evolution')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Population Diversity
    axes[0, 1].plot(iterations, diversity, 'purple', linewidth=2, marker='o', markersize=3)
    axes[0, 1].set_xlabel('Iteration')
    axes[0, 1].set_ylabel('Population Diversity')
    axes[0, 1].set_title('Population Diversity Over Time')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Add diversity trend line
    z = np.polyfit(iterations, diversity, 1)
    p = np.poly1d(z)
    axes[0, 1].plot(iterations, p(iterations), "r--", alpha=0.8, label='Trend')
    axes[0, 1].legend()
    
    # 3. Fitness Standard Deviation
    axes[1, 0].plot(iterations, fitness_std, 'orange', linewidth=2, marker='s', markersize=3)
    axes[1, 0].set_xlabel('Iteration')
    axes[1, 0].set_ylabel('Fitness Standard Deviation')
    axes[1, 0].set_title('Population Convergence (Lower Std = More Converged)')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Convergence Rate Analysis
    # Calculate improvement rate
    improvement_rates = []
    for i in range(1, len(best_fitness)):
        if best_fitness[i-1] > 0:
            rate = (best_fitness[i] - best_fitness[i-1]) / best_fitness[i-1]
            improvement_rates.append(rate)
        else:
            improvement_rates.append(0)
    
    rate_iterations = iterations[1:]
    axes[1, 1].plot(rate_iterations, improvement_rates, 'teal', linewidth=2, marker='^', markersize=3)
    axes[1, 1].axhline(y=0, color='red', linestyle='--', alpha=0.7)
    axes[1, 1].set_xlabel('Iteration')
    axes[1, 1].set_ylabel('Improvement Rate')
    axes[1, 1].set_title('Fitness Improvement Rate')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print convergence statistics
    print("\n" + "="*80)
    print("ABC CONVERGENCE ANALYSIS")
    print("="*80)
    
    print(f"Initial Population Fitness: {best_fitness[0]:.4f} ± {fitness_std[0]:.4f}")
    print(f"Final Population Fitness: {best_fitness[-1]:.4f} ± {fitness_std[-1]:.4f}")
    print(f"Total Fitness Improvement: {(best_fitness[-1] - best_fitness[0]):.4f}")
    print(f"Improvement Percentage: {((best_fitness[-1] - best_fitness[0]) / best_fitness[0] * 100):.1f}%")
    
    # Find convergence point (when improvement < 1% for 10 consecutive iterations)
    convergence_point = None
    for i in range(10, len(improvement_rates)):
        if all(abs(rate) < 0.01 for rate in improvement_rates[i-10:i]):
            convergence_point = iterations[i]
            break
    
    if convergence_point:
        print(f"Convergence Point: Iteration {convergence_point}")
        print(f"Convergence Rate: {convergence_point / len(iterations) * 100:.1f}% of total iterations")
    else:
        print("No clear convergence point detected within iteration limit")
    
    print(f"Initial Diversity: {diversity[0]:.4f}")
    print(f"Final Diversity: {diversity[-1]:.4f}")
    print(f"Diversity Loss: {((diversity[0] - diversity[-1]) / diversity[0] * 100):.1f}%")

# Visualize convergence
visualize_abc_convergence(result)

In [ ]:
# Parameter Sensitivity Analysis
def parameter_sensitivity_analysis():
    """Analyze sensitivity of ABC performance to different parameters"""
    
    print("Parameter Sensitivity Analysis...")
    
    # Test different colony sizes
    colony_sizes = [10, 20, 30, 40, 50]
    colony_results = []
    
    print("\nTesting Colony Sizes:")
    print(f"{'Colony Size':<12} {'Best Fitness':<12} {'Time (s)':<10} {'Diversity':<12}")
    print("-" * 50)
    
    for size in colony_sizes:
        optimizer = ArtificialBeeColonyOptimizer(
            zones=zones[:4],  # Use fewer zones for faster testing
            orders=orders[:15],  # Use fewer orders
            colony_size=size,
            max_iterations=50,  # Fewer iterations for faster testing
            limit=10
        )
        
        test_result = optimizer.optimize()
        final_diversity = test_result['diversity_history'][-1]['diversity']
        
        colony_results.append({
            'colony_size': size,
            'best_fitness': test_result['best_fitness'],
            'time': test_result['optimization_time'],
            'diversity': final_diversity
        })
        
        print(f"{size:<12} {test_result['best_fitness']:<12.4f} {test_result['optimization_time']:<10.2f} {final_diversity:<12.4f}")
    
    # Test different limit values
    limit_values = [5, 10, 15, 20, 25]
    limit_results = []
    
    print("\nTesting Abandonment Limits:")
    print(f"{'Limit':<8} {'Best Fitness':<12} {'Time (s)':<10} {'Scouts':<8}")
    print("-" * 42)
    
    for limit_val in limit_values:
        optimizer = ArtificialBeeColonyOptimizer(
            zones=zones[:4],
            orders=orders[:15],
            colony_size=20,
            max_iterations=50,
            limit=limit_val
        )
        
        test_result = optimizer.optimize()
        
        # Count scout bee activations (solutions replaced)
        scout_activations = sum(1 for fs in test_result['final_population'] if fs.trial_count == 0)
        
        limit_results.append({
            'limit': limit_val,
            'best_fitness': test_result['best_fitness'],
            'time': test_result['optimization_time'],
            'scouts': scout_activations
        })
        
        print(f"{limit_val:<8} {test_result['best_fitness']:<12.4f} {test_result['optimization_time']:<10.2f} {scout_activations:<8}")
    
    # Create sensitivity visualizations
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('ABC Parameter Sensitivity Analysis', fontsize=16, fontweight='bold')
    
    # Colony Size vs Fitness
    axes[0, 0].plot([r['colony_size'] for r in colony_results], 
                   [r['best_fitness'] for r in colony_results], 'o-', color='blue', linewidth=2)
    axes[0, 0].set_xlabel('Colony Size')
    axes[0, 0].set_ylabel('Best Fitness')
    axes[0, 0].set_title('Colony Size Impact on Solution Quality')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Colony Size vs Time
    axes[0, 1].plot([r['colony_size'] for r in colony_results], 
                   [r['time'] for r in colony_results], 's-', color='red', linewidth=2)
    axes[0, 1].set_xlabel('Colony Size')
    axes[0, 1].set_ylabel('Optimization Time (seconds)')
    axes[0, 1].set_title('Colony Size Impact on Computational Time')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Limit vs Fitness
    axes[1, 0].plot([r['limit'] for r in limit_results], 
                   [r['best_fitness'] for r in limit_results], 'o-', color='green', linewidth=2)
    axes[1, 0].set_xlabel('Abandonment Limit')
    axes[1, 0].set_ylabel('Best Fitness')
    axes[1, 0].set_title('Limit Impact on Solution Quality')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Limit vs Scout Activations
    axes[1, 1].plot([r['limit'] for r in limit_results], 
                   [r['scouts'] for r in limit_results], 's-', color='purple', linewidth=2)
    axes[1, 1].set_xlabel('Abandonment Limit')
    axes[1, 1].set_ylabel('Scout Bee Activations')
    axes[1, 1].set_title('Limit Impact on Exploration')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Run parameter sensitivity analysis
parameter_sensitivity_analysis()

In [ ]:
# Comparison with Previous Tiers
def compare_with_previous_tiers():
    """Compare ABC performance with backtracking and greedy baselines"""
    
    print("Comparison with Previous Tiers:")
    
    # Create smaller test problem for fair comparison
    test_zones = zones[:4]
    test_orders = orders[:12]
    
    # Run ABC on smaller problem
    abc_small = ArtificialBeeColonyOptimizer(
        zones=test_zones,
        orders=test_orders,
        colony_size=20,
        max_iterations=50,
        limit=10
    )
    
    abc_result = abc_small.optimize()
    
    # Simple greedy baseline
    def greedy_assignment(test_zones, test_orders):
        assignments = {}
        for order in test_orders:
            assignments[order.order_id] = {}
            for zone_id in order.items_by_zone:
                assignments[order.order_id][zone_id] = (order.order_id * 3) % 20
        return assignments
    
    greedy_solution = greedy_assignment(test_zones, test_orders)
    
    # Calculate metrics for all methods
    def calculate_solution_metrics(solution):
        total_time = 0
        zone_loads = defaultdict(float)
        
        for order in test_orders:
            if order.order_id in solution:
                completion_time = 0
                for zone_id, time_slot in solution[order.order_id].items():
                    if zone_id in order.items_by_zone:
                        zone = next(z for z in test_zones if z.zone_id == zone_id)
                        processing_time = order.items_by_zone[zone_id] * zone.pick_time
                        zone_loads[zone_id] += processing_time
                        completion_time = max(completion_time, time_slot)
                total_time += completion_time
        
        # Calculate balance
        if zone_loads:
            loads = list(zone_loads.values())
            balance = (max(loads) - min(loads)) / max(loads) if max(loads) > 0 else 0
        else:
            balance = 0
        
        throughput = len(test_orders) / max(total_time, 1)
        
        return {
            'completion_time': total_time,
            'balance': balance,
            'throughput': throughput
        }
    
    abc_metrics = calculate_solution_metrics(abc_result['best_solution'])
    greedy_metrics = calculate_solution_metrics(greedy_solution)
    
    # Create comparison visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('ABC vs Previous Tiers Comparison', fontsize=16, fontweight='bold')
    
    methods = ['ABC', 'Greedy']
    
    # Completion Time Comparison
    times = [abc_metrics['completion_time'], greedy_metrics['completion_time']]
    axes[0].bar(methods, times, color=['lightblue', 'lightcoral'], alpha=0.7)
    axes[0].set_ylabel('Total Completion Time')
    axes[0].set_title('Completion Time Comparison')
    axes[0].grid(True, alpha=0.3)
    
    # Balance Comparison
    balances = [abc_metrics['balance'], greedy_metrics['balance']]
    axes[1].bar(methods, balances, color=['lightgreen', 'lightyellow'], alpha=0.7)
    axes[1].set_ylabel('Workload Balance (Lower is Better)')
    axes[1].set_title('Balance Comparison')
    axes[1].grid(True, alpha=0.3)
    
    # Throughput Comparison
    throughputs = [abc_metrics['throughput'], greedy_metrics['throughput']]
    axes[2].bar(methods, throughputs, color=['gold', 'orange'], alpha=0.7)
    axes[2].set_ylabel('Throughput (Orders/Minute)')
    axes[2].set_title('Throughput Comparison')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print comparison table
    print("\n" + "="*70)
    print("PERFORMANCE COMPARISON SUMMARY")
    print("="*70)
    print(f"{'Method':<10} {'Completion':<12} {'Balance':<10} {'Throughput':<12} {'Fitness':<10}")
    print("-"*60)
    
    abc_fitness = abc_result['best_fitness']
    greedy_fitness = abc_small._calculate_fitness(greedy_solution)
    
    print(f"{'ABC':<10} {abc_metrics['completion_time']:<12.1f} {abc_metrics['balance']:<10.3f} "
          f"{abc_metrics['throughput']:<12.3f} {abc_fitness:<10.4f}")
    print(f"{'Greedy':<10} {greedy_metrics['completion_time']:<12.1f} {greedy_metrics['balance']:<10.3f} "
          f"{greedy_metrics['throughput']:<12.3f} {greedy_fitness:<10.4f}")
    
    # Calculate improvements
    time_improvement = ((greedy_metrics['completion_time'] - abc_metrics['completion_time']) / greedy_metrics['completion_time'] * 100)
    balance_improvement = ((greedy_metrics['balance'] - abc_metrics['balance']) / max(greedy_metrics['balance'], 0.001) * 100)
    throughput_improvement = ((abc_metrics['throughput'] - greedy_metrics['throughput']) / greedy_metrics['throughput'] * 100)
    fitness_improvement = ((abc_fitness - greedy_fitness) / greedy_fitness * 100)
    
    print(f"\nImprovement over Greedy:")
    print(f"  Completion Time: {time_improvement:.1f}%")
    print(f"  Balance: {balance_improvement:.1f}%")
    print(f"  Throughput: {throughput_improvement:.1f}%")
    print(f"  Overall Fitness: {fitness_improvement:.1f}%")

# Run comparison
compare_with_previous_tiers()

### Why This Tier Exists vs Previous Tiers

The Artificial Bee Colony algorithm provides a sophisticated metaheuristic approach that bridges the gap between exact methods and simple heuristics:

**Advantages over Tier 1 (Mathematical Formulation):**
- **Scalability**: Handles 25+ orders efficiently where MIP becomes computationally intractable
- **Population-Based Search**: Explores multiple solutions simultaneously, reducing risk of local optima
- **Adaptive Balance**: Automatically balances exploration (scout bees) and exploitation (employed/onlooker bees)
- **Robustness**: Less sensitive to parameter variations and problem structure
- **Parallelizable**: Natural parallel structure for distributed computing

**Advantages over Tier 2 (Backtracking):**
- **Population Diversity**: Maintains diverse solution population, avoiding single-path limitations
- **Global Search**: Better at escaping local optima through scout bee exploration
- **Faster Convergence**: Typically converges faster than systematic backtracking for large problems
- **Quality Assurance**: Probabilistic selection ensures good solutions are preserved and improved
- **Memory Efficiency**: Doesn't require extensive search tree storage

**Disadvantages vs Previous Tiers:**
- **No Optimality Guarantee**: Cannot prove solutions are mathematically optimal
- **Parameter Tuning**: Requires careful selection of colony size, limit, and iteration count
- **Stochastic Nature**: Different runs may produce different solutions
- **Convergence Uncertainty**: May not converge within iteration limit for difficult problems

### When to Use This Tier

- **Large-Scale Problems**: 20+ orders where exact methods are infeasible
- **Complex Landscapes**: Problems with many local optima where backtracking gets stuck
- **Robust Solutions**: When solution consistency across runs is important
- **Time-Critical Applications**: When good solutions are needed quickly
- **Research Prototyping**: Testing new problem formulations and objective functions

### Key Insights from ABC Approach

1. **Swarm Intelligence Effectiveness**: Colony collective behavior outperforms individual search strategies
2. **Exploration-Exploitation Balance**: Critical parameter affecting solution quality and convergence speed
3. **Population Diversity**: Essential for avoiding premature convergence to local optima
4. **Adaptive Abandonment**: Scout bee mechanism effectively reinitializes exhausted search regions
5. **Scalability Advantages**: Linear scalability with problem size compared to exponential backtracking

The ABC algorithm demonstrates how nature-inspired swarm intelligence can effectively solve complex zone balancing problems, providing high-quality solutions with reasonable computational requirements for large-scale applications.